# Step 2: the SAMap fit

SAMap (sc-samap 3.0.1) with default settings. numpy is seeded and hnswlib held to one thread, so the fit is repeatable.

In [ ]:
OUT = ""  # run folder
INPUTS = ""  # default: OUT/1_inputs (step 1)
MAPS = ""  # default: OUT/0_homology (step 0b)
N_ITERATIONS = 3
NCPUS = 8
SEED = 0  # numpy seed for the fit


## The fit

In [ ]:
import datetime
import os
from pathlib import Path

import h5py
import hnswlib
import numpy as np
from samap import SAMAP, save_samap
from samap.sam import SAM

assert OUT, "set OUT, the run folder"
inputs = {species: Path(INPUTS or Path(OUT) / "1_inputs") / f"{species}_fib_dec_all_raw.h5ad" for species in ("hs", "mm")}
maps = Path(MAPS or Path(OUT) / "0_homology")
out_dir = Path(OUT) / "2_samap"
partial_dir = Path(OUT) / "2_samap.partial"
assert not out_dir.exists(), f"{out_dir} exists; step 2 is done"
(partial_dir / "figures").mkdir(parents=True, exist_ok=True)
started = datetime.datetime.now()


def log(message):
    print(f"[{datetime.datetime.now():%H:%M:%S}] {message}", flush=True)


# SAMap draws from numpy's global random state without seeding it (SAMap issue #135), and hnswlib builds its
# indexes on several threads in a timing-dependent order. Seed numpy and hold every index to one thread.
_hnswlib_index = hnswlib.Index


def _single_thread_index(*args, **kwargs):
    index = _hnswlib_index(*args, **kwargs)
    index.set_num_threads(1)
    return index


hnswlib.Index = _single_thread_index
np.random.seed(SEED)


# One SAM per species, with SAMap's own default settings written out so they are visible.
PREPROCESS = dict(sum_norm="cell_median", norm="log", thresh_low=0.0, thresh_high=0.96, min_expression=1)
SAM_RUN = dict(preprocessing="StandardScaler", npcs=100, weight_PCs=False, k=20, n_genes=3000, weight_mode="rms",
               verbose=False)

sams = {}
for species, path in inputs.items():
    # SAM.load_data reads .raw.X instead of X when the file has .raw.
    with h5py.File(path, "r") as f:
        assert "raw" not in f, f"{path} has .raw, which SAM would read instead of X"
    sam = SAM()
    sam.load_data(str(path))
    log(f"{species}: {sam.adata.shape} from {path}")
    sam.preprocess_data(**PREPROCESS)
    sam.run(**SAM_RUN)
    log(f"{species}: SAM.run done")
    sams[species] = sam

# The fit. Twenty cross-species neighbors and correlation-scaled edge weights are SAMAP.run's defaults.
np.random.seed(SEED)
sm = SAMAP(sams, f_maps=f"{maps}/", save_processed=False)
log("SAMAP init done")
sm.run(n_iterations=N_ITERATIONS, ncpus=NCPUS)
log("SAMAP run done")

## Save the fit

In [ ]:
import json

import pandas as pd

save_samap(sm, str(partial_dir / "samap.pkl"))
log(f"wrote {partial_dir / 'samap.pkl'}")

joint = sm.samap.adata
for column in ("sample", "timepoint", "cell_type_annotations", "cell_subtype_annotations"):
    values = pd.Series(index=joint.obs_names, dtype=object)
    for sam in sm.sams.values():
        obs = sam.adata.obs
        shared = obs.index.intersection(joint.obs_names)
        values.loc[shared] = obs.loc[shared, column].astype(str).to_numpy()
    joint.obs[column] = values.to_numpy()

# Object columns mixing text and NaN write poorly to h5ad.
for column in joint.obs.columns:
    if joint.obs[column].dtype == object:
        joint.obs[column] = joint.obs[column].astype(str).replace("nan", "NA")

joint.uns["provenance"] = json.dumps({
    "created": datetime.datetime.now().isoformat(),
    "step": "samap step 2 (notebooks/2_samap.ipynb)",
    "inputs": {species: str(path) for species, path in inputs.items()},
    "X": ("SAM-preprocessed expression: log2(cell_median-normalized + 1) with min_expression 1; block-diagonal "
          "over species; NOT raw counts"),
    "raw_counts": "the step-1 inputs (X = raw counts)",
    "graphs": "obsp['connectivities'] = stitched joint kNN; varp = homology graphs",
})
joint.write_h5ad(partial_dir / "samap_joint.h5ad")
log(f"wrote {partial_dir / 'samap_joint.h5ad'}: {joint.shape}, obs {list(joint.obs.columns)}")

## Evaluation

A failure here is recorded in `eval.json` and does not stop the step.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scipy.sparse as sp
from samap.analysis import get_mapping_scores
from samap.core.mapping import _avg_as
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def sample_mixing(graph, samples):
    """Observed and expected share of neighbor weight in another sample."""
    graph = graph.tocsr()
    n = graph.shape[0]
    categories, codes = np.unique(samples, return_inverse=True)
    onehot = sp.csr_matrix((np.ones(n), (np.arange(n), codes)), shape=(n, categories.size))
    per_sample = graph @ onehot
    total = np.asarray(per_sample.sum(1)).ravel()
    own = np.asarray(per_sample[np.arange(n), codes]).ravel()
    has_neighbors = total > 0
    observed = ((total - own) / np.maximum(total, 1e-12))[has_neighbors]
    counts = np.bincount(codes, minlength=categories.size).astype(float)
    expected = (1.0 - counts[codes] / n)[has_neighbors]
    return {
        "observed_other_sample_frac": float(observed.mean()),
        "expected_other_sample_frac": float(expected.mean()),
        "mixing_ratio": float(observed.mean() / max(expected.mean(), 1e-12)),
        "n_samples": int(categories.size),
    }


def knn_label_agreement(graph, labels):
    """Mean share of neighbor weight on nuclei with the nucleus's own label."""
    graph = graph.tocsr()
    n = graph.shape[0]
    categories, codes = np.unique(labels, return_inverse=True)
    onehot = sp.csr_matrix((np.ones(n), (np.arange(n), codes)), shape=(n, categories.size))
    per_label = graph @ onehot
    total = np.asarray(per_label.sum(1)).ravel()
    own = np.asarray(per_label[np.arange(n), codes]).ravel()
    has_neighbors = total > 0
    return float((own[has_neighbors] / total[has_neighbors]).mean())

In [ ]:
graph = joint.obsp["connectivities"].tocsr()
species = joint.obs["species"].to_numpy()
evaluation = {"n_cells_joint": int(joint.n_obs)}

alignment = _avg_as(sm.samap)
evaluation["alignment"] = {r: {c: float(alignment.loc[r, c]) for c in alignment.columns} for r in alignment.index}
evaluation["mean_alignment"] = float(alignment.to_numpy()[~np.eye(alignment.shape[0], dtype=bool)].mean())
log(f"alignment scores: {evaluation['alignment']}; mean {evaluation['mean_alignment']:.3f}")

symmetric = ((graph + graph.T) * 0.5).tocsr()
sc.tl.leiden(joint, adjacency=symmetric, key_added="joint_leiden", resolution=1.0, flavor="igraph", n_iterations=2,
             directed=False)
log(f"joint leiden: {joint.obs['joint_leiden'].nunique()} clusters")

for sid in np.unique(species):
    is_species = species == sid
    within = graph[is_species][:, is_species]
    obs = sm.sams[sid].adata.obs.loc[joint.obs_names[is_species]]
    samples = obs["sample"].astype(str).to_numpy()
    cell_type = obs["cell_type_annotations"].astype(str).to_numpy()
    subtype = obs["cell_subtype_annotations"].astype(str).to_numpy()
    leiden = joint.obs["joint_leiden"].to_numpy()[is_species]
    numbers = {
        "sample_mixing": sample_mixing(within, samples),
        "knn_celltype_agreement": knn_label_agreement(within, cell_type),
        "leiden_vs_celltype_ARI": float(adjusted_rand_score(cell_type, leiden)),
        "leiden_vs_celltype_NMI": float(normalized_mutual_info_score(cell_type, leiden)),
        "leiden_vs_sample_NMI": float(normalized_mutual_info_score(samples, leiden)),
    }
    if len(set(subtype)) > 1:
        numbers["knn_subtype_agreement"] = knn_label_agreement(within, subtype)
        numbers["leiden_vs_subtype_ARI"] = float(adjusted_rand_score(subtype, leiden))
        numbers["leiden_vs_subtype_NMI"] = float(normalized_mutual_info_score(subtype, leiden))
    evaluation[f"within_{sid}"] = numbers
    print(sid, json.dumps(numbers))

# SAMap's mapping scores between label groups of the two species.
evaluation["errors"] = []
for tag, keys in [
    ("celltype", {"hs": "cell_type_annotations", "mm": "cell_type_annotations"}),
    ("subtype", {"hs": "cell_type_annotations", "mm": "cell_subtype_annotations"}),
    ("timepoint", {"hs": "timepoint", "mm": "timepoint"}),
]:
    try:
        _, scores = get_mapping_scores(sm, keys, n_top=0)
        scores.to_csv(partial_dir / f"mapping_{tag}.csv")
        if tag == "celltype":
            human_rows = [i for i in scores.index if i.startswith("hs_")]
            mouse_columns = [c for c in scores.columns if c.startswith("mm_")]
            evaluation["celltype_mapping"] = {r: {c: float(scores.loc[r, c]) for c in mouse_columns}
                                              for r in human_rows}
    except Exception as error:
        evaluation["errors"].append(f"mapping scores {tag}: {error!r}")
        log(f"mapping scores {tag} failed (not fatal): {error!r}")
print(json.dumps(evaluation.get("celltype_mapping"), indent=1))

In [ ]:
# UMAPs of the joint object. A failure here does not stop the step.
try:
    for column in ("timepoint", "sample", "cell_type_annotations"):
        values = pd.Series(index=joint.obs_names, dtype=object)
        for sid, sam in sm.sams.items():
            obs = sam.adata.obs
            shared = obs.index.intersection(joint.obs_names)
            values.loc[shared] = sid + ":" + obs.loc[shared, column].astype(str)
        joint.obs[f"joint_{column}"] = pd.Categorical(values.to_numpy())
    for column, name in [("species", "umap_species.png"), ("joint_timepoint", "umap_timepoint.png"),
                         ("joint_sample", "umap_sample.png"), ("joint_cell_type_annotations", "umap_celltype.png")]:
        fig = sc.pl.embedding(joint, basis="X_umap", color=column, return_fig=True, show=False)
        fig.savefig(partial_dir / "figures" / name, dpi=150, bbox_inches="tight")
        plt.close(fig)
except Exception as error:
    evaluation["errors"].append(f"figures: {error!r}")
    log(f"figures failed (not fatal): {error!r}")
(partial_dir / "eval.json").write_text(json.dumps(evaluation, indent=2) + "\n")
if evaluation["errors"]:
    log(f"evaluation errors, also in eval.json: {evaluation['errors']}")

## Record settings, checksums and versions

In [ ]:
import hashlib
import platform
from importlib.metadata import version


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


meta = {
    "arm": "vanilla_raw (stock SAMap on raw counts)",
    "started": started.isoformat(timespec="seconds"),
    "finished": datetime.datetime.now().isoformat(timespec="seconds"),
    "host": platform.node(),
    "inputs": {species: {"path": str(path), "sha256": sha256_of(path)} for species, path in inputs.items()},
    "maps": {name: {"path": str(maps / "hsmm" / name), "sha256": sha256_of(maps / "hsmm" / name)}
             for name in ("hs_to_mm.txt", "mm_to_hs.txt")},
    "preprocess": PREPROCESS,
    "sam_run": SAM_RUN,
    "n_iterations": N_ITERATIONS,
    "ncpus": NCPUS,
    "seed": SEED,
    "hnswlib_threads": 1,
    "mean_alignment": evaluation["mean_alignment"],
    "versions": {package: version(package) for package in
                 ("sc-samap", "anndata", "scanpy", "numpy", "scipy", "hnswlib", "umap-learn", "scikit-learn")},
}
(partial_dir / "meta.json").write_text(json.dumps(meta, indent=2) + "\n")
print(json.dumps(meta, indent=1))

os.replace(partial_dir, out_dir)
log(f"wrote {out_dir}: {sorted(p.name for p in out_dir.iterdir())}")